# Linear Probe — how strong is the tree-encoding signal, really?

**The question.** Every result so far has been produced by a 2.44M-parameter TabResNet, so any of it can be blamed on the network: maybe the trunk is doing the work, maybe the capacity lets it exploit anything. Replace the network with a **plain linear classifier** and that objection disappears. Whatever a linear model scores *is* the signal sitting in the input.

### Two kinds of linear model, on purpose
| | what it is | what it rules out |
|---|---|---|
| **logistic regression** (`run_linear_probe.py`) | scikit-learn, convex objective, solved to the global optimum | no learning rate, no epochs, no training dynamics — nothing left to argue about |
| **linear head** (`--fusion linear`) | the same `run_fusion.py` pipeline with the trunk replaced by one `nn.Linear` | directly comparable to every TabResNet number we have, same recipe, same splits |

If the two agree, the answer is solid. If the SGD one is lower, it simply has not converged, and the converged number is the truth.

### Three inputs
| input | features | asks |
|---|---|---|
| `x` | 10 | how far do the raw features go on their own? |
| **`tree`** | ~5,400 | **the bits ALONE — no raw features, no hidden layer** |
| `x+tree` | ~5,410 | both together |

The `tree`-only arm is the one that answers your question. A linear model on bits alone has no way to invent anything: every bit is a yes/no answer to one split question, and the model can only weight them and add them up.

### How to read it
- **`tree` alone lands near the tree ceiling (~0.845)** → the encoding carries essentially the whole signal, and the network was never the source of the result.
- **`tree` alone lands well below it** → the bits need a nonlinear model to be useful, and the TabResNet is earning its place.
- **`x+tree` ≈ `tree`** → the raw features add nothing once the bits are present.

Same data and protocol as every other experiment: credit (OpenML 361055), OOB-honest hard bits, 100 trees of depth 6, stratified split.

**On the regularisation strength `C`.** `C = 1/lambda`, so a *small* `C` means *strong* regularisation. With ~5,400 bits and ~11,200 training rows the penalty does real work, and the best value sits at the strongly regularised end. The grid runs `1e-4 ... 1.0` in roughly 3x steps and is chosen on validation, never on test. The script also warns if the winner lands on the edge of the grid, because that means the true optimum may lie outside it and the reported score would be a lower bound.

⏱ **~30 minutes.** Runtime → GPU → Run all (the GPU only helps the SGD arms; the logistic regression is CPU).

In [ ]:
# 1 · mount Drive + get the code
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/TKCE/linear_probe'
os.makedirs(DRIVE, exist_ok=True)
%cd /content
!git clone https://github.com/sushanedulloo/TKCE.git 2>/dev/null || echo 'already cloned'
%cd /content/TKCE
!git pull
print('results ->', DRIVE)

In [ ]:
# 2 · install deps
!pip install -q openml catboost optuna

In [ ]:
# 2b · OPTIONAL — only if OpenML 504s: upload openml_cache_clean5.tar.gz (else press Cancel)
import os, glob, tarfile
dst = '/root/.cache/openml/org/openml/www'
if glob.glob(dst + '/tasks/361055'):
    print('credit already cached — skip')
else:
    try:
        from google.colab import files; files.upload()
    except Exception as e: print('skipped:', e)
    hits = glob.glob('/content/**/openml_cache_clean5.tar.gz', recursive=True)
    if hits:
        os.makedirs(dst, exist_ok=True)
        with tarfile.open(hits[0]) as t: t.extractall(dst)
        print('cache extracted')
    else: print('no bundle — will use OpenML directly')

In [ ]:
# 3 · THE DEFINITIVE ONE — converged logistic regression on x / tree / x+tree
#     Convex objective: scikit-learn returns the global optimum, so there is no
#     optimiser to blame. C is chosen on validation. ~10-20 min (CPU).
!python -u run_linear_probe.py --task 361055 --out results/linear_probe
import shutil; shutil.copytree('results/linear_probe', f'{DRIVE}/probe', dirs_exist_ok=True)

In [ ]:
# 4 · the same thing through the normal pipeline: trunk replaced by one nn.Linear
#     Same recipe as every TabResNet run, so the numbers are directly comparable.
base = ('--task 361055 --encoding oob --ensemble 2 --epochs 400 --dropout 0 --l1 0 '
        '--weight-decay 1e-3 --lr 3e-4 --batch-size 128 --device auto --log-batches')
!python -u run_fusion.py {base} --fusion linear --views "x;tree;x+tree" --out results/fusion/lin_sgd
shutil.copytree('results/fusion/lin_sgd', f'{DRIVE}/lin_sgd', dirs_exist_ok=True)

In [ ]:
# 5 · the reference: the full-capacity TabResNet on the same split
!python -u run_fusion.py {base} --fusion tabresnet --views "x;tree;x+tree" --out results/fusion/lin_ref
shutil.copytree('results/fusion/lin_ref', f'{DRIVE}/tabresnet_ref', dirs_exist_ok=True)

In [ ]:
# 6 · the comparison table
import json, glob, pandas as pd
rows = []
p = json.load(open(glob.glob('results/linear_probe/linear_probe_*.json')[0]))
CEIL = p['tree_ceiling']
for r in p['results']:
    rows.append(dict(model='logistic regression (converged)', input=r['view'],
                     features=r['n_features'], params=r['n_features'] * 2 + 2,
                     train_auc=r['train_auc'], test_auc=r['test_auc']))
for tag, lab in [('lin_sgd', 'linear head (SGD)'), ('lin_ref', 'TabResNet (2.44M)')]:
    js = glob.glob(f'results/fusion/{tag}/fusion_*.json')
    js = [f for f in js if 'epochs' not in f and 'batches' not in f]
    if not js: continue
    s = json.load(open(js[0]))
    e = pd.read_csv(glob.glob(f'results/fusion/{tag}/fusion_*_epochs.csv')[0])
    for r in s['results']:
        g = e[e.model == r['model']].groupby('epoch').train_auc.mean()
        rows.append(dict(model=lab, input=r['views'], features=r['concat_dim'],
                         params=r.get('params'), train_auc=round(g.iloc[-1], 4),
                         test_auc=r['test_auc']))
t = pd.DataFrame(rows)
print(t.round(4).to_string(index=False))
print(f'\nbest tree (LightGBM): {CEIL:.4f}')
tree_only = t[t.input == 'tree']
print('\n--- the headline: bits alone, no raw features ---')
for _, r in tree_only.iterrows():
    print(f"  {r.model:32s} {r.test_auc:.4f}   ({r.test_auc/CEIL:.1%} of the tree ceiling)")
t.to_csv(f'{DRIVE}/linear_probe_summary.csv', index=False)

In [ ]:
# 7 · one picture: test AUC by model and input
import matplotlib.pyplot as plt, numpy as np
BLUE, ORANGE, AQUA = '#2a78d6', '#eb6834', '#1baf7a'
MUTED, GRID = '#52514e', '#c8c8c4'
order = ['x', 'tree', 'x+tree']
models = list(dict.fromkeys(t.model))
cols = {models[i]: c for i, c in enumerate([BLUE, ORANGE, AQUA][:len(models)])}
fig, a = plt.subplots(figsize=(10, 5.2))
w = 0.8 / len(models)
for i, m in enumerate(models):
    sub = t[t.model == m].set_index('input').reindex(order)
    xs = np.arange(len(order)) + i * w - 0.4 + w / 2
    a.bar(xs, sub.test_auc, width=w * 0.92, color=cols[m], label=m, edgecolor='white', lw=1.2)
    for x, v in zip(xs, sub.test_auc):
        if v == v: a.text(x, v + 0.002, f'{v:.4f}', ha='center', fontsize=8.5)
a.axhline(CEIL, ls='--', color=MUTED, lw=1.4, label=f'best tree {CEIL:.4f}')
a.set_xticks(range(len(order)))
a.set_xticklabels(['x\n(10 raw features)', 'tree\n(bits ONLY)', 'x + tree\n(both)'])
a.set_ylabel('test AUC'); a.set_ylim(0.7, 0.88)
a.set_title('How much signal is in the tree encoding?', fontweight='bold', loc='left')
a.legend(fontsize=9, frameon=False); a.grid(alpha=.25, axis='y')
plt.tight_layout(); plt.savefig(f'{DRIVE}/linear_probe.png', dpi=150); plt.show()

In [ ]:
# 8 · loss / AUC curves for the SGD linear arms
from plot_training import plot_training_curves
plot_training_curves('results/fusion/lin_sgd', model='tree',
                     out=f'{DRIVE}/lin_sgd_tree_curves.png')
plot_training_curves('results/fusion/lin_ref', model='tree',
                     out=f'{DRIVE}/tabresnet_tree_curves.png')